In [1]:
from spin_lattices import KagomeLattice, SpinLattice, ChainLattice, SquareLattice, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from misc_utils import make_unpacked_configurations
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nqs_playground_helpers import (
    SamplingOptions,
    split_into_batches,
    safe_exp,
    sample_exactly,
    sample_full,
    forward_with_batches,
)
from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
import sys
from kagome_cnn import KagomeCNNRegression
from torch.nn.utils import parameters_to_vector
import time
from slater_determinant import SlaterDeterminant
from vmc_amplitude import (
    compute_local_energies,
    find_nbd,
    find_nbd_reference,
    apply_diag_to_basis_states,
    apply_off_diag_to_basis_states,
    true_relsigns,
    almost_true_relsigns,
)

2023-08-03 20:32:57.877 | DEBUG    | lattice_symmetries:__init__:50 - Initializing Haskell runtime...
2023-08-03 20:32:57.880 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-08-03 20:32:57.928 | DEBUG    | lattice_symmetries:__init__:54 - Setting Python exception handler...
set_python_exception_handler ...


In [2]:
lattice = KagomeLattice(2, 3)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=0.5,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
matrix, states = find_nbd(system.hamiltonian, system.basis.states[:100])
matrix_ref, states_ref = find_nbd_reference(system.hamiltonian, system.basis.states[:100])
assert np.allclose(matrix.toarray(), matrix_ref.toarray())
assert np.allclose(states, states_ref)

2023-08-03 20:33:01.071 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=18
2023-08-03 20:33:01.073 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-03 20:33:01.089 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 48620
/vol/tcm10/ischurov/frustrations-eda/vmc_amplitude.py:92: ComplexWarning: Casting complex values to real discards the imaginary part
  coeffs_data = np.concatenate(coeff_rows).astype(np.float64)


In [3]:
class LogProbDenseNet(nn.Module):
    def __init__(self, system: SpinSystem, n_hidden: int = 100, hidden_layers=1):
        super().__init__()
        self.system = system
        self.n_hidden = n_hidden
        self.hidden_layers = hidden_layers
        layers = [nn.Linear(system.number_spins, n_hidden), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(n_hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(
            torch.from_numpy(
                make_unpacked_configurations(x, self.system.number_spins).astype(np.float32)
            )
        )

In [4]:
def test_full_energy():
    lattice = ChainLattice(10)
    system = HeisenbergJ1J2(lattice, J1=1, ground_state_cache_dir=Path("groundstates"))
    true_energy, ground_state = system.get_eigenstates(1)
    true_energy = true_energy[0]
    ground_state = ground_state[:, 0]

    states = system.canonical_basis.states

    def true_logprob(states: npt.NDArray[np.uint64]) -> npt.NDArray[np.float64]:
        return 2 * np.log(np.abs(system.get_ground_state_coeffs(states))) # type ignore
    
    E_loc = compute_local_energies(
        system.hamiltonian,
        states,
        true_relsigns(system),
        true_logprob,
    )

    assert np.isclose(E_loc @ (ground_state ** 2), true_energy)

test_full_energy()

2023-08-03 20:33:01.205 | DEBUG    | heisenberg_hamiltonians:__init__:444 - use_symmetries is None and lattice is not in symmetries whitelist, setting use_symmetries=False
2023-08-03 20:33:01.207 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=10
2023-08-03 20:33:01.209 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-03 20:33:01.218 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 252
2023-08-03 20:33:01.224 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-ChainLattice10x1-1.0-1.0-False-None-1.pickle
2023-08-03 20:33:01.226 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -18.0617854180


IndexError: boolean index did not match indexed array along dimension 0; dimension is 252 but corresponding boolean dimension is 1652

In [ ]:
def test_local_E_loc_consistency():
    lattice = ChainLattice(10)
    system = HeisenbergJ1J2(lattice, J1=1, ground_state_cache_dir=Path("groundstates"))
    system.get_eigenstates(1)

    def true_logprob(states: npt.NDArray[np.uint64]) -> npt.NDArray[np.float64]:
        return 2 * np.log(np.abs(system.get_ground_state_coeffs(states)))  # type ignore

    E_loc_full = compute_local_energies(
        system.hamiltonian,
        system.canonical_basis.states,
        true_relsigns(system),
        true_logprob,
    )

    for size in range(1, len(system.canonical_basis.states), 10):
        print(f"{size=}", end="...")
        sampled_states = np.random.choice(system.canonical_basis.states, size=size)
        E_loc_sampled = compute_local_energies(
            system.hamiltonian,
            sampled_states,
            true_relsigns(system),
            true_logprob,
        )

        assert np.allclose(
            E_loc_full[np.searchsorted(system.canonical_basis.states, sampled_states)],
            E_loc_sampled,
        )

        print("ok")


test_local_E_loc_consistency()

2023-08-03 20:23:23.168 | DEBUG    | heisenberg_hamiltonians:__init__:444 - use_symmetries is None and lattice is not in symmetries whitelist, setting use_symmetries=False
2023-08-03 20:23:23.169 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=10
2023-08-03 20:23:23.171 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-03 20:23:23.180 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 252
2023-08-03 20:23:23.184 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-ChainLattice10x1-1.0-1.0-False-None-1.pickle
2023-08-03 20:23:23.186 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -18.0617854180


size=1...ok
size=11...ok
size=21...ok
size=31...ok
size=41...ok
size=51...ok
size=61...ok
size=71...ok
size=81...ok
size=91...ok
size=101...ok
size=111...ok
size=121...ok
size=131...ok
size=141...ok
size=151...ok
size=161...ok
size=171...ok
size=181...ok
size=191...ok
size=201...ok
size=211...ok
size=221...ok
size=231...ok
size=241...ok
size=251...ok


In [13]:
# Calculate the gradient norms
# FROM: GPT-4
def get_gradient_norm(parameters):
    grads = []
    for p in parameters:
        if p.grad is not None:
            grads.append(p.grad)
    grad_vector = parameters_to_vector(grads)
    return torch.linalg.norm(grad_vector)

# END FROM

In [14]:
def differentiable_safe_exp(x: Tensor, normalise: bool = True) -> Tensor:
    r"""Calculate ``exp(x)`` avoiding overflows. Result is not equal to
    ``exp(x)``, but rather proportional to it. If ``normalise==True``, then
    this function makes sure that output tensor elements sum up to 1.
    """
    x = x - torch.max(x)
    x = torch.exp(x)
    if normalise:
        x = x / torch.sum(x)
    return x

In [67]:
# n_samples = 48620
n_samples = 100
lr = 1e-2
momentum = 0.0
batch_size = 64
sign_noise = 0.0
weight_decay = 0
annealing_steps = 0  # 400
initial_temp = 3
sampling_mode = "exact"

lattice = TriangleLattice(6, 4)
# lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1.2,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
true_energy, _ = system.get_eigenstates(1)
true_energy = true_energy[0]

if len(system.canonical_basis.states) > 50000:
    eval_set = np.random.choice(system.canonical_basis.states, 50000, replace=False)
else:
    eval_set = system.canonical_basis.states

log_prob_fn = LogProbDenseNet(system, n_hidden=512, hidden_layers=1)
# log_prob_fn = SlaterAbs(
#     system.lattice,
#     system.canonical_basis,
#     sign_cache_dir=Path("signs_cache"),
#     initialization="randn",
# )

# log_prob_fn = KagomeCNNRegression(system.lattice, hidden_channels1=32, hidden_channels2=64)
# optimizer = torch.optim.SGD(log_prob_fn.parameters(), lr=lr, momentum=momentum)
optimizer = torch.optim.Adam(log_prob_fn.parameters(), lr=lr, weight_decay=weight_decay)

true_amplitudes = torch.from_numpy(np.abs(system.get_ground_state_coeffs(eval_set)))
relsigns_fn = almost_true_relsigns(system, eps=sign_noise)

writer = SummaryWriter(
    log_dir=(
        f"experiments/{datetime.now().strftime('%Y_%m_%d')}/{datetime.now().strftime('%H_%M_%S')}"
    )
)

sampling_time = 0
local_energies_time = 0
optimization_time = 0
evaluation_time = 0

for step in range(10000):
    sampling_time_tick = time.time()
    if sampling_mode == "exact":
        states, log_probs, all_probs = sample_exactly(
            log_prob_fn,
            system.basis,
            SamplingOptions(
                number_samples=n_samples,
                number_chains=1,
                mode="exact",
                sweep_size=1,
                number_discarded=0,
            ),
            return_all_probs=True,
        )
        states, weights = torch.unique(states.view(-1), return_counts=True)
        weights = weights.float() / torch.sum(weights)

    elif sampling_mode == "full":
        states, log_probs, _extra = sample_full(
            log_prob_fn,
            system.basis,
            SamplingOptions(
                number_samples=1,
                number_chains=1,
                mode="full",
                sweep_size=1,
                number_discarded=0,
            ),
        )
        states = states.view(-1)
        weights = _extra["weights"].view(-1)
        all_probs = weights
    else:
        raise ValueError(f"Unknown sampling mode: {sampling_mode}")
    sampling_time += time.time() - sampling_time_tick

    ipr = torch.sum(all_probs**2)
    writer.add_scalar("loss/ipr", ipr, step)

    local_energies_time_tick = time.time()
    E = compute_local_energies(
        system.hamiltonian,
        states.detach().numpy(),
        relsigns_fn=relsigns_fn,
        log_prob_fn=lambda s: log_prob_fn(torch.from_numpy(s)).view(-1).detach().numpy(),
    )
    E = torch.from_numpy(E).to(torch.float32)
    local_energies_time += time.time() - local_energies_time_tick

    # states = states.view(-1, states.size(-1))
    # log_probs = log_probs.view(-1)
    # weights = weights.view(-1)

    # Compute output gradient
    sampling_time_tick = time.time()

    with torch.no_grad():
        grad = 4 * (E - E @ weights) * weights
        # coeff 4 is due to: 2 from formula, 2 due to we are working with log probs
        # instead of log amplitudes

        grad = grad.view(-1, 1)
        grad_norm = torch.linalg.norm(grad)
        #    logger.info("‖∇E‖₂ = {}", grad_norm)
        writer.add_scalar("loss/‖∇E‖₂", grad_norm, step)
        writer.add_scalar("loss/E_variance", grad_norm / n_samples, step)

        # Calculate full energy
        if sampling_mode == "exact":
            E_full = E @ safe_exp(log_prob_fn(states).view(-1), normalise=True)
            writer.add_scalar("loss/E_full", E_full - torch.tensor(true_energy), step)

    optimizer.zero_grad()
    # batch_size = self.config.inference_batch_size

    # Computing gradients for the amplitude network
    # logger.info("Computing gradients...")
    # if _should_optimize(self.config.amplitude):
    #     self.config.amplitude.train()
    forward_fn = log_prob_fn
    for states_chunk, grad_chunk in split_into_batches((states.view(-1, 1), grad), batch_size):
        output = forward_fn(states_chunk.view(-1))
        output.backward(grad_chunk, retain_graph=True)

        if step < annealing_steps:
            temp = initial_temp * (1 - step / annealing_steps)
            probs = differentiable_safe_exp(output, normalise=True)
            entropy = -torch.sum(probs * torch.log(probs))
            entropy_loss = -temp * entropy
            entropy_loss.backward()

    full_gradient_norm = get_gradient_norm(forward_fn.parameters())
    writer.add_scalar("loss/full_gradient_norm", full_gradient_norm, step)

    optimizer.step()
    optimization_time += time.time() - sampling_time_tick

    evaluation_time_tick = time.time()
    predicted_amplitudes = safe_exp(log_prob_fn(eval_set) * 0.5)

    overlap = find_overlap(true_amplitudes, predicted_amplitudes)
    writer.add_scalar("overlap", overlap, step)
    logger.info(
        f"{step}: overlap = {overlap:.3f}, ‖∇E‖₂ = {grad_norm:.3f}, full_gradient_norm = {full_gradient_norm:.3f}"
    )
    evaluation_time += time.time() - evaluation_time_tick

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkSize: 4167
[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 649; batchedOperatorChunkS

KeyboardInterrupt: 

In [28]:
print(f"{sampling_time=}, {local_energies_time=}, {optimization_time=}, {evaluation_time=}")

sampling_time=32.43007731437683, local_energies_time=40.43660497665405, optimization_time=5.056471586227417, evaluation_time=61.717533349990845


In [46]:
log_prob_fn(states)

tensor([3.3361e-06, 7.0296e-06, 5.7566e-06, 5.2540e-06, 8.3943e-06, 8.2055e-07,
        3.5701e-06, 8.5260e-06, 1.0833e-05, 2.2178e-06, 5.5872e-06, 1.9821e-06,
        2.5439e-06, 2.9751e-06, 1.9589e-06, 1.3938e-05, 5.9044e-07, 2.6833e-07,
        1.1206e-06, 4.6837e-07, 2.4727e-06, 1.1445e-05, 1.0944e-06, 1.3824e-06,
        2.2380e-06, 2.2174e-08, 5.1166e-06, 1.0813e-05, 6.4515e-06, 8.6632e-06,
        2.2583e-06, 1.8863e-06, 1.2750e-06, 8.2732e-08, 6.3106e-06, 1.2767e-05,
        6.0862e-07, 7.8696e-08, 1.1487e-05, 4.7970e-08, 2.3524e-05, 1.3037e-05,
        1.0291e-07, 7.0273e-06, 1.1669e-05, 1.3252e-06, 6.5573e-06, 5.9186e-07,
        6.1990e-06, 5.8048e-07, 2.3535e-05, 1.7368e-05, 1.3051e-06, 1.4308e-06,
        9.3598e-06, 4.8032e-06, 7.4175e-07, 3.7684e-06, 1.1660e-05, 3.6536e-05,
        3.3422e-05, 1.4347e-06, 4.1040e-05, 5.0391e-07, 2.2022e-05, 4.2902e-09,
        3.5087e-06, 7.4986e-07, 8.6881e-07, 7.5847e-06, 7.7746e-07, 5.4851e-07,
        1.0073e-06, 4.4313e-06, 1.0937e-

In [30]:
system.canonical_basis.states.shape

(48620,)

In [32]:
(eval_set == system.canonical_basis.states).all()

True

In [33]:
gs = np.asarray(system.get_ground_state_coeffs(system.basis.states), order='C')
np.allclose(system.hamiltonian @ gs, system.eigenvalues[0] * gs)

[Debug]   [LOCALE0]   remoteBufferSize: 150000; numChunks: 240; batchedOperatorChunkSize: 203


True